<h1>Creación de Modelos Predictivos</h1>

<h2>Importar las Librerías Necesarias</h2>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)

<h2>Modelos de Clasificación</h2>

In [2]:
dataset = fetch_ucirepo(id=468)
df_clasificacion = pd.concat([dataset.data.features, dataset.data.targets], axis=1)

<h3>Análisis de la Fuente de Datos</h3>
<p>Antes de iniciar el proceso de preparación es necesario conocer ciertos aspectos de la fuente de datos. En este caso se evaluaron los siguientes aspectos para saber el nivel de calidad de los datos existentes y si es necesario aplicar alguna transformación para realizar correcciones a alguno de los datos.</p>
<b>Consideraciones de Análisis</b>
<ul>
    <li>Validar la existencia de los valores nulos.</li>
</ul>

In [3]:
print("Cantidad de valores vacios en cada columna:")
df_clasificacion.isna().sum()

Cantidad de valores vacios en cada columna:


Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64

<h3>Fundamentación del Modelo</h3>

In [4]:
X_clasificacion = df_clasificacion.drop(columns='Revenue')
y_clasificacion = df_clasificacion['Revenue'].astype(int)

In [5]:
X_trainclass, X_testclass, y_trainclass, y_testclass = train_test_split(X_clasificacion, y_clasificacion, test_size=0.2, random_state=42, stratify=y_clasificacion)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
sns.set_theme(style="whitegrid")

<h3>Preparación mínima para los modelos</h3>
<p>Siguiendo lo indicado en la práctica, las variables numéricas se escalan con <code>StandardScaler</code> y las categóricas se convierten en dummies (<code>OneHotEncoder</code>). Ambos pasos se incluyen dentro de un <code>Pipeline</code> junto con el modelo, para que se ajusten solo con los datos de entrenamiento y, en la validación cruzada, se reajusten en cada fold (sin fuga de información). <code>OperatingSystems</code>, <code>Browser</code>, <code>Region</code> y <code>TrafficType</code> se tratan como categóricas porque son códigos, no cantidades.</p>

In [ ]:
columnas_categoricas = ['Month', 'VisitorType', 'Weekend', 'OperatingSystems', 'Browser', 'Region', 'TrafficType']
columnas_numericas = [c for c in X_clasificacion.columns if c not in columnas_categoricas]
print("Numéricas:", columnas_numericas)
print("Categóricas:", columnas_categoricas)

def crear_preprocesador():
    return ColumnTransformer(transformers=[
        ('num', StandardScaler(), columnas_numericas),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='if_binary', sparse_output=False), columnas_categoricas)
    ])

<h3>Construcción de los Modelos</h3>
<p>Se construyen dos modelos con <code>scikit-learn</code> sobre la partición 80 % / 20 % estratificada realizada arriba (<code>stratify=y</code>, <code>random_state=42</code>):</p>
<ul>
    <li><b>Naive Bayes Gaussiano:</b> modelo probabilístico que asume independencia condicional entre variables y distribución normal en cada una. Es rápido y simple, pero esas suposiciones se violan aquí (variables correlacionadas, dummies binarias y distribuciones sesgadas).</li>
    <li><b>Árbol de Decisión:</b> particiona el espacio de variables con reglas del tipo "si <i>PageValues</i> &gt; x". Es interpretable y captura relaciones no lineales. Sin límites tiende a sobreajustarse, por eso se restringe con <code>max_depth=5</code> y <code>min_samples_leaf=20</code>.</li>
</ul>

In [ ]:
modelo_nb = Pipeline([('prep', crear_preprocesador()), ('clf', GaussianNB())])
modelo_dt = Pipeline([('prep', crear_preprocesador()),
                      ('clf', DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=42))])

modelo_nb.fit(X_trainclass, y_trainclass)
modelo_dt.fit(X_trainclass, y_trainclass)

pred_nb = modelo_nb.predict(X_testclass)
pred_dt = modelo_dt.predict(X_testclass)

<h3>Evaluación: Matrices de Confusión y Métricas</h3>
<p>Para una matriz de confusión con la clase 1 (compra) como positiva:</p>
<ul>
    <li><b>Exactitud</b> = (TP + TN) / (TP + TN + FP + FN)</li>
    <li><b>Precisión</b> = TP / (TP + FP)</li>
    <li><b>Recall</b> = TP / (TP + FN)</li>
    <li><b>F1</b> = 2 · Precisión · Recall / (Precisión + Recall)</li>
</ul>

In [ ]:
def metricas_manuales(y_real, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_real, y_pred).ravel()
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
            'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1': f1}

def comparar_manual_vs_sklearn(nombre, y_real, y_pred):
    m = metricas_manuales(y_real, y_pred)
    sk = {'Accuracy': accuracy_score(y_real, y_pred), 'Precision': precision_score(y_real, y_pred),
          'Recall': recall_score(y_real, y_pred), 'F1': f1_score(y_real, y_pred)}
    tabla = pd.DataFrame({'Manual': {k: m[k] for k in sk}, 'scikit-learn': sk})
    tabla['Diferencia'] = (tabla['Manual'] - tabla['scikit-learn']).abs()
    print(f"=== {nombre} ===")
    print(f"TN={m['TN']}  FP={m['FP']}  FN={m['FN']}  TP={m['TP']}")
    display(tabla.round(4))
    return m

m_nb = comparar_manual_vs_sklearn("Naive Bayes", y_testclass, pred_nb)
m_dt = comparar_manual_vs_sklearn("Árbol de Decisión", y_testclass, pred_dt)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, nombre, pred in zip(axes, ['Naive Bayes', 'Árbol de Decisión'], [pred_nb, pred_dt]):
    ConfusionMatrixDisplay.from_predictions(y_testclass, pred, display_labels=['No compra', 'Compra'],
                                            cmap='Blues', ax=ax, colorbar=False)
    ax.set_title(f"Matriz de confusión - {nombre}")
plt.tight_layout()
plt.show()

In [ ]:
print("Naive Bayes\n", classification_report(y_testclass, pred_nb, target_names=['No compra', 'Compra']))
print("Árbol de Decisión\n", classification_report(y_testclass, pred_dt, target_names=['No compra', 'Compra']))

<h4>Interpretación de las matrices de confusión</h4>
<p><i>(Complétese con los números obtenidos al ejecutar.)</i> Guía para redactar:</p>
<ul>
    <li><b>TP:</b> sesiones con compra que el modelo detectó. <b>TN:</b> sesiones sin compra correctamente descartadas.</li>
    <li><b>FP:</b> el modelo predijo compra y no la hubo. <b>FN:</b> el modelo predijo "no compra" pero el usuario sí compró.</li>
    <li>Compárese cuántos FP y FN comete cada modelo: normalmente Naive Bayes tiende a marcar muchas sesiones como compra (más FP, recall alto pero precisión baja), mientras que el árbol es más equilibrado y con mayor exactitud.</li>
    <li>Recuérdese que un modelo que siempre predice "no compra" tendría una exactitud cercana al 84 %, por lo que la exactitud sola no basta; hay que mirar precisión, recall y F1 de la clase "compra".</li>
</ul>

<h3>Validación Cruzada Estratificada (5 folds)</h3>
<p>Se aplica sobre los datos de entrenamiento únicamente (el conjunto de prueba queda reservado). Al usar <code>Pipeline</code>, el escalado y las dummies se reajustan dentro de cada fold, evitando fuga de información.</p>

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'accuracy': 'accuracy', 'precision': 'precision', 'recall': 'recall', 'f1': 'f1'}

resultados_cv = {}
for nombre, modelo in [('Naive Bayes', modelo_nb), ('Árbol de Decisión', modelo_dt)]:
    cv = cross_validate(modelo, X_trainclass, y_trainclass, cv=skf, scoring=scoring)
    resultados_cv[nombre] = cv
    print(f"--- {nombre} ---")
    por_fold = pd.DataFrame({m: cv[f'test_{m}'] for m in scoring}, index=[f'Fold {i}' for i in range(1, 6)])
    por_fold.loc['Media'] = por_fold.mean()
    por_fold.loc['Desv. est.'] = por_fold.iloc[:5].std()
    display(por_fold.round(4))

In [ ]:
resumen_cv = pd.DataFrame({
    nombre: {m: f"{cv[f'test_{m}'].mean():.3f} ± {cv[f'test_{m}'].std():.3f}" for m in scoring}
    for nombre, cv in resultados_cv.items()
})
resumen_cv

<h4>Comentario de la validación cruzada</h4>
<p><i>(Complétese con los resultados.)</i> Puntos a comentar: (1) si las medias de CV son similares a las métricas en prueba, el modelo generaliza bien; (2) una desviación estándar baja indica estabilidad entre folds; (3) si el árbol rinde mucho mejor en entrenamiento que en CV habría sobreajuste; (4) cuál modelo es más estable y en cuál métrica difieren más.</p>

<h3>Comparación de Modelos y Selección</h3>

In [ ]:
comparacion = pd.DataFrame({
    'Naive Bayes': {k: m_nb[k] for k in ['Accuracy', 'Precision', 'Recall', 'F1']},
    'Árbol de Decisión': {k: m_dt[k] for k in ['Accuracy', 'Precision', 'Recall', 'F1']}
}).T
display(comparacion.round(4))

comparacion.plot(kind='bar', figsize=(9, 5), rot=0)
plt.title("Comparación de métricas en el conjunto de prueba")
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.show()

In [ ]:
# Interpretabilidad del árbol: variables más importantes y estructura (primeros niveles)
arbol = modelo_dt.named_steps['clf']
nombres_feat = modelo_dt.named_steps['prep'].get_feature_names_out()
importancias = pd.Series(arbol.feature_importances_, index=nombres_feat).sort_values(ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(22, 7), gridspec_kw={'width_ratios': [1, 2]})
importancias[::-1].plot(kind='barh', ax=axes[0], title="Top 10 importancias (Árbol de Decisión)")
plot_tree(arbol, max_depth=3, feature_names=nombres_feat, class_names=['No compra', 'Compra'],
          filled=True, fontsize=8, ax=axes[1])
axes[1].set_title("Árbol (primeros 3 niveles)")
plt.tight_layout()
plt.show()

<h4>Modelo seleccionado y justificación</h4>
<p><i>(Confirmar con los resultados obtenidos.)</i> Lo habitual con este dataset es que el <b>Árbol de Decisión</b> supere claramente a Naive Bayes en exactitud, precisión y F1, y sea más estable en la validación cruzada. Además, es interpretable: sus reglas muestran que <code>PageValues</code> es el factor dominante, lo que permite explicar la predicción a un equipo de negocio. Naive Bayes sufre porque su supuesto de independencia no se cumple (<code>BounceRates</code>/<code>ExitRates</code> correlacionadas, muchas variables dummy). Si Naive Bayes mostrara un recall notablemente más alto, habría que ponderarlo según el costo del negocio (ver siguiente pregunta). Se selecciona el modelo con mejor F1 y estabilidad en CV, justificándolo con los números de las tablas anteriores.</p>

<h3>Pregunta de Análisis</h3>
<p><b>¿Qué consecuencias podrían tener los falsos positivos y los falsos negativos en el contexto del problema? ¿Qué métrica es más importante y por qué?</b></p>
<ul>
    <li><b>Falso positivo (se predice compra y no ocurre):</b> la tienda gasta recursos en una sesión que no iba a convertir: descuentos innecesarios (pérdida de margen), chats o pop-ups intrusivos que pueden molestar al usuario, o campañas de remarketing mal dirigidas. También puede generar una visión demasiado optimista de la demanda (p. ej., planificar inventario de más).</li>
    <li><b>Falso negativo (se predice no compra y sí compra, o habría comprado con un pequeño incentivo):</b> se pierde la oportunidad de intervenir en usuarios con alta intención de compra (no ofrecer ayuda, envío gratis o recordatorios), con pérdida directa de ingresos. Si el modelo se usa para descartar sesiones, también se subestima la demanda.</li>
    <li><b>Métrica prioritaria:</b> dado que la clase de interés es minoritaria (~15 %), la exactitud es engañosa. Si las intervenciones son baratas (mostrar un banner, enviar un correo), el costo de un FN (venta perdida) supera al de un FP, por lo que priorizaríamos el <b>recall</b> de la clase "compra" sin dejar que la precisión se desplome. Si las intervenciones son costosas (descuentos fuertes), se priorizaría la <b>precisión</b>. Como equilibrio general, el <b>F1-score</b> de la clase positiva es la métrica de referencia más razonable. Con ese criterio, el umbral de decisión también puede ajustarse (p. ej. con <code>predict_proba</code>) según el costo real de cada error.</li>
</ul>